# Sparse Metric Anchors — reproduce every number in the paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/635jack/sparse-metric-anchors/blob/main/colab/reproduce.ipynb)

Companion notebook to *Sparse Metric Anchors for a Single-View 3D Generative Prior: The
Output Frame Is the Bottleneck* (ISIR, Sorbonne Université, 2026).

**What this does, on a CPU runtime, in about two minutes:** it downloads the benchmark
(`jack635/sparse-metric-anchors-ycb`) and the code (`635jack/sparse-metric-anchors`),
then recomputes **every table and figure of the paper from the published campaign
results** — with an independent implementation of the statistics — and re-runs the one
result that needs no generative model, the Poisson witness, live.

**What it does not do:** generate shapes. The backbone (`WaLa-SV-1B`) needs about 13 GB of
RAM to load and an 18 GB checkpoint; that is a high-RAM or Pro runtime, and the last
section says how.

Every cell prints the paper's value next to the recomputed one.

In [ ]:
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "huggingface_hub", "open3d", "scipy", "matplotlib"], check=True)

In [ ]:
from pathlib import Path
import subprocess, os, time
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import HfHubHTTPError

# The campaign results and the contact sets live in the code repository itself, so one
# git clone brings them all at once. Only the five reference meshes the Poisson witness
# needs come from the dataset -- five files, fetched one at a time with a back-off on
# the Hub's anonymous rate limit (HTTP 429). A full anonymous download of the 680-file
# dataset trips that limit; do not do it from here.
if not Path("sparse-metric-anchors").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/635jack/sparse-metric-anchors"], check=True)
REPO = Path("sparse-metric-anchors").resolve()
sys.path[:0] = [str(REPO / "tools"), str(REPO)]
RES = REPO / "results"
CONTACTS = REPO / "data" / "contacts"

TEST_OBJECTS = ["002_master_chef_can", "006_mustard_bottle", "011_banana", "025_mug", "035_power_drill"]
MESH = {}
for o in TEST_OBJECTS:
    for attempt in range(6):
        try:
            MESH[o] = Path(hf_hub_download("jack635/sparse-metric-anchors-ycb", f"objects/{o}/mesh.obj", repo_type="dataset"))
            break
        except HfHubHTTPError as e:
            if "429" not in str(e) or attempt == 5: raise
            time.sleep(20 * (attempt + 1))
print("code + results :", REPO); print("results :", sorted(p.name for p in RES.glob("*.json")))
print("meshes :", len(MESH), "from the Hub")

In [ ]:
import json, numpy as np
from math import comb

def sign_test(d):
    # two-sided exact sign test, ties discarded -- the paper's protocol
    d = [x for x in d if x != 0]; n = len(d); k = sum(1 for x in d if x > 0)
    p = min(1.0, 2 * sum(comb(n, i) for i in range(0, min(k, n - k) + 1)) / 2 ** n)
    return k, n, p

def gains(fn, cond, base="baseline", key="f2"):
    d = json.load(open(RES / fn))
    return {(o, s): d[o][f"{cond}_s{s}"][key] - d[o][f"{base}_s{s}"][key]
            for o in d for s in (0, 1, 2)
            if f"{cond}_s{s}" in d[o] and f"{base}_s{s}" in d[o]
            and key in d[o][f"{cond}_s{s}"] and key in d[o][f"{base}_s{s}"]}

def baseline(fn, key="f2"):
    d = json.load(open(RES / fn))
    return {(o, s): d[o][f"baseline_s{s}"][key] for o in d for s in (0, 1, 2) if f"baseline_s{s}" in d[o]}

def show(label, value, paper):
    print(f"  {label:44s} recomputed {value:>10}   paper {paper}")

## Table I — constraint anchoring, 42 objects, frame estimated

In [ ]:
g = gains("full42_combined.json", "palpation8_128"); v = np.array(list(g.values()))
b = np.array(list(baseline("full42_combined.json").values()))
k, n, p = sign_test(v)
per_obj = {}
for (o, s), x in g.items(): per_obj.setdefault(o, []).append(x)
show("unanchored baseline, F@2 %", f"{b.mean():.1f}", "53.1")
show("mean gain", f"{v.mean():+.2f}", "+3.21")
show("median / std", f"{np.median(v):+.2f} / {v.std(ddof=1):.2f}", "+1.69 / 7.52")
show("positive cases (ties discarded)", f"{k} / {n}", "83 / 125")
show("sign test, two-sided", f"{p:.1e}", "3.1e-04")
show("objects with positive mean gain", f"{sum(np.mean(x) > 0 for x in per_obj.values())} / {len(per_obj)}", "35 / 42")
go = gains("full42_oracle.json", "palpation8_128")
show("oracle ceiling (frame given)", f"{np.mean(list(go.values())):+.2f}", "+7.82")

## Table II — the two anchoring routes, and their combination

In [ ]:
mono = json.load(open(RES / "full42_combined.json")); dm4 = json.load(open(RES / "dm4_resultats.json"))
rows = []
for o in mono:
    if o not in dm4: continue
    for s in (0, 1, 2):
        try:
            rows.append((mono[o][f"baseline_s{s}"]["f2"], mono[o][f"palpation8_128_s{s}"]["f2"],
                         dm4[o][f"baseline_s{s}"]["f2"], dm4[o][f"guide_s{s}"]["f2"]))
        except KeyError: pass
rb, rt, db, dt = np.array(rows).T
show("image only", f"{rb.mean():.1f}", "53.1")
show("  + contacts as constraint", f"{rt.mean():.1f}", "56.3")
show("depth reprojected to 4 views", f"{db.mean():.1f}", "64.3")
show("  + contacts as constraint", f"{dt.mean():.1f}", "64.7")
show("per-case oracle choice of route", f"{np.maximum(rt, dt).mean():.1f}", "72.2")
gd = db - rb
show("reprojection gain, paired mean", f"{gd.mean():+.2f}", "+11.22")
show("cases improved / lost", f"{(gd > 0).sum()} / {(gd < 0).sum()}", "87 / 39")
show("mean gain where improved / where lost", f"{gd[gd > 0].mean():+.1f} / {gd[gd < 0].mean():+.1f}", "+26.1 / -22.1")
k, n, p = sign_test(dt - db)
show("contacts on top of anchored frame", f"{(dt - db).mean():+.2f}, {k}/{n}, p={p:.3f}", "+0.38, 75/126, p=0.040")
show("corr(reprojection gain, contact gain)", f"{np.corrcoef(gd, rt - rb)[0, 1]:+.3f}", "+0.056")

## Figure 2a — seven controlled manipulations of the anchor set

In [ ]:
def paired(a, b):
    keys = sorted(set(a) & set(b)); return [a[k] - b[k] for k in keys]
dh = {c: gains("dh116_regime.json", c) for c in ["dh8", "dh8n", "dh8n_bruit", "dh4"]}
st = {c: gains("strategies.json", c) for c in ["fb", "rl"]}
comps = [
  ("8 patches instead of 4",       paired(gains("palpation_combined.json", "palpation_128"), gains("sites_4.json", "palpation4_128")), "+2.22, 12/15, p=0.035"),
  ("32 patches instead of 8",      paired(gains("sites_32.json", "palpation32_128"), gains("palpation_combined.json", "palpation_128")), "-0.24, 5/15, p=0.302"),
  ("512 anchors instead of 32",    paired(gains("guidance_campaign.json", "occluded_512"), gains("guidance_campaign.json", "occluded_32")), "-0.16, 8/15, p=1.000"),
  ("surface normals added",        paired(dh["dh8n"], dh["dh8"]), "+0.93, 10/15, p=0.302"),
  ("anchors off the hidden face",  paired(st["rl"], st["fb"]), "+0.82, 11/15, p=0.118"),
  ("placement noise 3 mm / 15 deg", paired(dh["dh8n_bruit"], dh["dh8n"]), "-2.40, 3/15, p=0.035"),
  ("4 zones instead of 8",         paired(dh["dh4"], dh["dh8"]), "-2.55, 3/15, p=0.035"),
]
for lab, v, paper in comps:
    k, n, p = sign_test(v); show(lab, f"{np.mean(v):+.2f}, {k}/{n}, p={p:.3f}", paper)

## Figure 2b — contact visibility does not predict the gain

In [ ]:
import matplotlib.pyplot as plt
d = json.load(open(RES / "strategies.json")); pts = []
for o in d:
    for c in ("fb", "lr", "rl"):
        for s in (0, 1, 2):
            g = d[o][f"{c}_s{s}"]; pts.append((g["part_occultee"], g["f2"] - d[o][f"baseline_s{s}"]["f2"]))
occ, gain = np.array(pts).T
show("correlation(occluded fraction, gain), 45 cases", f"{np.corrcoef(occ, gain)[0, 1]:+.3f}", "-0.145")
zero = gain[occ == 0]
show("gain with ZERO anchors on the hidden face", f"{zero.mean():+.2f} over {len(zero)} cases", "+5.68 over 3")
a, b = np.polyfit(occ, gain, 1)
plt.figure(figsize=(4.5, 4)); plt.scatter(occ, gain, s=14, alpha=.7); plt.scatter(occ[occ == 0], zero, s=40, c="g")
x = np.linspace(0, .85, 2); plt.plot(x, a * x + b, "r-")
plt.axhline(0, c="k", lw=.5); plt.xlabel("fraction of anchors occluded"); plt.ylabel("gain, F@2 %"); plt.title(f"r = {np.corrcoef(occ, gain)[0,1]:+.3f}"); plt.show()

## Section III — the output pose is resampled with the noise

In [ ]:
pd_ = json.load(open(RES / "pose_determinism.json")); a = pd_["agrege"]
show("asymmetric objects: n, median inter-seed rotation", f"{a['asymetriques']['n']}, {a['asymetriques']['ecart_median_deg']:.1f} deg", "16, 131.8 deg in this run; 136.5 ± 3.7 over four runs")
show("symmetric objects:  n, median inter-seed rotation", f"{a['symetriques']['n']}, {a['symetriques']['ecart_median_deg']:.1f} deg", "26, 134.4 deg in this run; 135.7 ± 3.7 over four runs")
show("stable below 30 deg: asym / sym", f"{a['asymetriques']['stables_sous_30']} / {a['symetriques']['stables_sous_30']}", "1 / 0")
show("corr(asymmetry, stability)", f"{a['correlation_asymetrie_stabilite']:+.3f}", "-0.124 in this run; -0.14 ± 0.06 over four runs")

## Section VI-G — run-to-run variance: frame re-estimated vs frozen

In [ ]:
for fn, lab in [("plancher.json", "frame re-estimated"), ("plancher_gel.json", "frame frozen")]:
    r = json.load(open(RES / fn))
    t = np.array([x["temoin"] for x in r]); g = np.array([x["guide"] for x in r]); dd = g - t
    show(f"{lab}: sd unanchored / anchored / paired diff", f"{t.std(ddof=1):.2f} / {g.std(ddof=1):.2f} / {dd.std(ddof=1):.2f}",
         "0.49 / 2.29 / 2.50" if "gel" not in fn else "0.68 / 0.70 / 1.29")
sa = np.std([x["guide"] for x in json.load(open(RES / "plancher.json"))], ddof=1)
sb = np.std([x["guide"] for x in json.load(open(RES / "plancher_gel.json"))], ddof=1)
show("share of anchored variance due to frame re-estimation", f"{100 * (sa**2 - sb**2) / sa**2:.0f} %", "91 % (banana; frozen-frame sd equals unanchored sd on 3 more objects)")

## Table III — the Poisson witness, run live

Six oriented contacts reconstructed by screened Poisson, scored with the paper's own
alignment and metric. This one is *recomputed*, not re-read: its value moves by a few
tenths between runs because both Poisson and the alignment sample points at random
(the paper measures the scoring alone at $\sigma = 0.32$).

In [ ]:
import open3d as o3d
from eval_fusion import load_and_normalize_mesh, align_prediction_to_gt, compute_chamfer_and_fscore
from poisson_temoin import poisson, OBJECTS, STRATEGIES
npz = np.load(CONTACTS / "strategies.npz", allow_pickle=True)
out = {"6": [], "18": []}
for obj in OBJECTS:
    gt, _ = load_and_normalize_mesh(MESH[obj])
    for lab, strats in [("6", ["front_back"]), ("18", STRATEGIES)]:
        pos = np.concatenate([npz[f"{obj}|{s}|pos"] for s in strats]); nrm = np.concatenate([npz[f"{obj}|{s}|nrm"] for s in strats])
        best = max((compute_chamfer_and_fscore(gt, align_prediction_to_gt(gt, m))["f_scores"]["2.0%"]["f_score"]
                    for m in (poisson(pos, nrm, dp) for dp in (3, 4, 5, 6)) if m is not None), default=float("nan"))
        out[lab].append(best)
show("Poisson, 6 oriented contacts, mean F@2 %", f"{np.nanmean(out['6']):.2f}", "19.90 ± 0.36 over five runs")
show("Poisson, 18 oriented contacts", f"{np.nanmean(out['18']):.2f}", "20.44 ± 0.18 over five runs")
show("image only, same objects (from Table I data)", f"{np.mean([baseline('full42_combined.json')[(o, s)] for o in OBJECTS for s in (0,1,2)]):.2f}", "56.52")

## What a GPU runtime adds

Everything above re-reads or recomputes results. To *generate* — to see the constraint
act on $\hat z_0$ — you need the backbone. On a high-RAM runtime (Colab Pro, or any
machine with ≥ 16 GB RAM and a GPU):

```bash
cd sparse-metric-anchors && pip install -r requirements.txt
python tools/guided_sampling.py \
    --image  <DATA>/objects/011_banana/image.png \
    --contacts <DATA>/contacts/011_banana/palpation8_128.pt \
    --out_dir out/demo --name 011_banana --guidance_weight 0.05 --seeds 0
```

The first call downloads `ADSKAILab/WaLa-SV-1B` (18 GB). A 20-step guided sample takes
about a minute on an M2 Max; `--guidance_weight 0` reproduces the unanchored baseline
exactly, which is the paired control every campaign uses.